# Assignment 2 — Triangle Rasterisation & Z-buffering

> **GAMES101 — Intro to Computer Graphics** (Lingqi Yan, UCSB).
> Course site: <https://sites.cs.ucsb.edu/~lingqi/teaching/games101.html>
>
> The course ships C++ starter code with Eigen + OpenCV. I'm doing the same tasks in
> Python notebooks so I can iterate on the math cell-by-cell. Notes at the top of each
> notebook are what I actually needed to remember to get the assignment out.

## Topic

Now that we can transform vertices, we have to actually **fill** the triangle with
pixels. The trick has two parts:

1. **Bounding-box + inside test.** Walk every pixel inside the triangle's axis-aligned
   bounding box and keep it if the three cross products all agree in sign — i.e. the
   pixel is on the same side of every edge.
2. **Per-pixel depth via barycentrics.** Interpolate `z` at each covered pixel and
   compare against a **z-buffer** so that whichever triangle is closer wins.

![z buffer](https://upload.wikimedia.org/wikipedia/commons/4/4e/Z_buffer.svg)
*Z-buffering: the front pixel wins per-fragment (Wikipedia).*

![barycentric](https://upload.wikimedia.org/wikipedia/commons/b/b7/TriangleBarycentricCoordinates.svg)
*Barycentric coordinates on a triangle (Wikipedia).*

See: [Rasterisation](https://en.wikipedia.org/wiki/Rasterisation), [Z-buffering](https://en.wikipedia.org/wiki/Z-buffering), [Barycentric coordinate system](https://en.wikipedia.org/wiki/Barycentric_coordinate_system).

### Task
- `inside_triangle(x, y, v)`
- `rasterize_triangle` with barycentric depth
- Bonus: **MSAA 2x2**, and fix the black seam on the shared edge.


In [1]:
import numpy as np

def inside_triangle(x, y, v):
    # v: list of 3 (x,y) points, cross-product sign test
    def cross(a, b, c):
        return (b[0]-a[0])*(c[1]-a[1]) - (b[1]-a[1])*(c[0]-a[0])
    p = (x, y)
    s1 = cross(v[0], v[1], p)
    s2 = cross(v[1], v[2], p)
    s3 = cross(v[2], v[0], p)
    return (s1 > 0 and s2 > 0 and s3 > 0) or (s1 < 0 and s2 < 0 and s3 < 0)


In [2]:
def rasterize_triangle(t, W, H, framebuffer, zbuffer, color):
    v = [(p[0], p[1]) for p in t]
    z = [p[2] for p in t]
    xs = [p[0] for p in v]; ys = [p[1] for p in v]
    x0, x1 = int(np.floor(min(xs))), int(np.ceil(max(xs)))
    y0, y1 = int(np.floor(min(ys))), int(np.ceil(max(ys)))
    for y in range(max(0, y0), min(H, y1)):
        for x in range(max(0, x0), min(W, x1)):
            if inside_triangle(x + 0.5, y + 0.5, v):
                # TODO barycentric depth interp -- for now just take z[0]
                if z[0] < zbuffer[y, x]:
                    zbuffer[y, x] = z[0]
                    framebuffer[y, x] = color


This produces the right silhouettes but the overlap is wrong -- I'm not interpolating depth per pixel. Have to add barycentrics next.